Optimización de Hiperparámetros y Serialización


## 1. Importación de Librerías

In [ ]:
import os, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, mean_absolute_error,
    mean_squared_error, r2_score
)

print("Librerías importadas correctamente.")

## 2. Cargar el Dataset Limpio

In [ ]:
!wget https://raw.githubusercontent.com/FIJI-1919/TRABAJO_PROGRAMACION_CIENCIA/refs/heads/main/data/dataset_clientes_limpio.csv

In [ ]:
data_limpio = pd.read_csv("dataset_clientes_limpio.csv")

## 3. Definición de Modelos y Funciones

In [ ]:
def get_classification_models() -> dict:
    return {
        "LogisticRegression": Pipeline(steps=[
            ("classifier", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
        ]),
        "DecisionTreeClassifier": Pipeline(steps=[
            ("classifier", DecisionTreeClassifier(random_state=42, class_weight="balanced")),
        ]),
        "SVM": Pipeline(steps=[
            ("classifier", SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")),
        ]),
    }

def get_regression_models() -> dict:
    return {
        "LinearRegression":     Pipeline(steps=[("regressor", LinearRegression())]),
        "DecisionTreeRegressor": Pipeline(steps=[("regressor", DecisionTreeRegressor(random_state=42))]),
    }

def train_and_fit_model(pipeline, X_train, y_train):
    pipeline.fit(X_train, y_train)
    return pipeline

def evaluate_classification(y_true, y_pred, y_prob=None) -> dict:
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        metricas["ROC-AUC"] = roc_auc_score(y_true, y_prob)
    return metricas

def evaluate_regression(y_true, y_pred) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R²": r2_score(y_true, y_pred)}

def tune_classification_model(pipeline, param_grid, X_train, y_train, method='grid', n_iter=10):
    """
    Optimiza hiperparámetros de un clasificador.
    - method='grid'   → GridSearchCV (exhaustivo, grillas pequeñas).
    - method='random' → RandomizedSearchCV (eficiente, grillas grandes como SVM).
    Usa StratifiedKFold para mantener proporción de clases en cada fold.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    if method == 'grid':
        buscador = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=0)
    else:
        buscador = RandomizedSearchCV(pipeline, param_grid, n_iter=n_iter,
                                      cv=cv, scoring='f1', n_jobs=-1, random_state=42, verbose=0)
    buscador.fit(X_train, y_train)
    print(f"  Mejor F1 (CV): {buscador.best_score_:.4f}")
    print(f"  Mejores params: {buscador.best_params_}")
    return buscador.best_estimator_

def tune_regression_model(pipeline, param_grid, X_train, y_train, method='grid', n_iter=10):
    """
    Optimiza hiperparámetros de un regresor.
    Usa KFold estándar (apropiado para targets continuos).
    """
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    if method == 'grid':
        buscador = GridSearchCV(pipeline, param_grid, cv=cv, scoring='r2', n_jobs=-1, verbose=0)
    else:
        buscador = RandomizedSearchCV(pipeline, param_grid, n_iter=n_iter,
                                      cv=cv, scoring='r2', n_jobs=-1, random_state=42, verbose=0)
    buscador.fit(X_train, y_train)
    print(f"  Mejor R² (CV): {buscador.best_score_:.4f}")
    print(f"  Mejores params: {buscador.best_params_}")
    return buscador.best_estimator_

print("Funciones definidas.")

## 4. Particiones Train / Test

In [ ]:
SEMILLA = 42

X_clf = df.drop(columns=['abandono'])
y_clf = df['abandono']
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEMILLA, stratify=y_clf
)

X_reg = df.drop(columns=['gasto_mensual', 'abandono'])
y_reg = df['gasto_mensual']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=SEMILLA
)
print("Particiones listas.")

## 5. Métricas Base (Referencia)

In [ ]:
clf_base = {n: train_and_fit_model(p, X_train_clf, y_train_clf) for n, p in get_classification_models().items()}
reg_base = {n: train_and_fit_model(p, X_train_reg, y_train_reg) for n, p in get_regression_models().items()}

res_base_clf = {}
for nombre, modelo in clf_base.items():
    y_pred = modelo.predict(X_test_clf)
    y_prob = modelo.predict_proba(X_test_clf)[:, 1] if hasattr(modelo, 'predict_proba') else None
    res_base_clf[nombre] = evaluate_classification(y_test_clf, y_pred, y_prob)

res_base_reg = {}
for nombre, modelo in reg_base.items():
    res_base_reg[nombre] = evaluate_regression(y_test_reg, modelo.predict(X_test_reg))

print("=== CLASIFICADORES BASE ===")
display(pd.DataFrame(res_base_clf).T.round(4))
print("\n=== REGRESORES BASE ===")
display(pd.DataFrame(res_base_reg).T.round(2))

## 6. Optimización de Clasificadores

### ¿Por qué StratifiedKFold?
Mantiene la proporción de clases en cada fold — evita folds sin la clase minoritaria y genera estimaciones más estables.

### ¿GridSearch vs. RandomizedSearch?
- **GridSearchCV**: prueba todas las combinaciones — ideal cuando la grilla es pequeña.
- **RandomizedSearchCV**: muestrea aleatoriamente — eficiente para espacios grandes como SVM.

### 6.1 Regresión Logística — GridSearchCV

In [ ]:
malla_lr = {'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0]}

print("Optimizando LogisticRegression...")
mejor_lr = tune_classification_model(
    get_classification_models()['LogisticRegression'],
    malla_lr, X_train_clf, y_train_clf, method='grid'
)

### 6.2 Árbol de Decisión Clasificador — GridSearchCV

In [ ]:
malla_dt_clf = {
    'classifier__max_depth':        [3, 5, 7, 10],
    'classifier__min_samples_split': [2, 5, 10]
}

print("Optimizando DecisionTreeClassifier...")
mejor_dt_clf = tune_classification_model(
    get_classification_models()['DecisionTreeClassifier'],
    malla_dt_clf, X_train_clf, y_train_clf, method='grid'
)

### 6.3 SVM — RandomizedSearchCV

In [ ]:
malla_svm = {
    'classifier__C':     [0.1, 1.0, 10.0, 100.0],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1, 1.0]
}

print("Optimizando SVM (RandomizedSearchCV)...")
mejor_svm = tune_classification_model(
    get_classification_models()['SVM'],
    malla_svm, X_train_clf, y_train_clf, method='random', n_iter=8
)

## 7. Optimización de Regresores

### 7.1 Regresión Lineal (sin hiperparámetros relevantes en baseline)

In [ ]:
mejor_lr_reg = train_and_fit_model(get_regression_models()['LinearRegression'], X_train_reg, y_train_reg)
print("LinearRegression entrenado (sin optimización).")

### 7.2 Árbol de Regresión — GridSearchCV

In [ ]:
malla_dt_reg = {
    'regressor__max_depth':        [3, 5, 7, 10, None],
    'regressor__min_samples_split': [2, 5, 10]
}

print("Optimizando DecisionTreeRegressor...")
mejor_dt_reg = tune_regression_model(
    get_regression_models()['DecisionTreeRegressor'],
    malla_dt_reg, X_train_reg, y_train_reg, method='grid'
)

## 8. Comparación: Base vs. Optimizado

In [ ]:
mejores_clf = {'LogisticRegression': mejor_lr, 'DecisionTreeClassifier': mejor_dt_clf, 'SVM': mejor_svm}
mejores_reg = {'LinearRegression': mejor_lr_reg, 'DecisionTreeRegressor': mejor_dt_reg}

res_opt_clf = {}
for nombre, modelo in mejores_clf.items():
    y_pred = modelo.predict(X_test_clf)
    y_prob = modelo.predict_proba(X_test_clf)[:, 1] if hasattr(modelo, 'predict_proba') else None
    res_opt_clf[nombre] = evaluate_classification(y_test_clf, y_pred, y_prob)

res_opt_reg = {}
for nombre, modelo in mejores_reg.items():
    res_opt_reg[nombre] = evaluate_regression(y_test_reg, modelo.predict(X_test_reg))

print("=== CLASIFICADORES OPTIMIZADOS ===")
display(pd.DataFrame(res_opt_clf).T.round(4))
print("\n=== REGRESORES OPTIMIZADOS ===")
display(pd.DataFrame(res_opt_reg).T.round(2))

### 8.1 Impacto de la Optimización — Gráfico Comparativo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

nombres_clf = list(res_base_clf.keys())
f1_base = [res_base_clf[n]['F1-Score'] for n in nombres_clf]
f1_opt  = [res_opt_clf[n]['F1-Score']  for n in nombres_clf]
x = range(len(nombres_clf))
axes[0].bar([i - 0.2 for i in x], f1_base, width=0.4, label='Base',      color='steelblue')
axes[0].bar([i + 0.2 for i in x], f1_opt,  width=0.4, label='Optimizado', color='coral')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(nombres_clf, rotation=15)
axes[0].set_title('F1-Score: Base vs. Optimizado', fontweight='bold')
axes[0].set_ylabel('F1-Score')
axes[0].legend()
axes[0].set_ylim(0, 1)

nombres_reg = list(res_base_reg.keys())
r2_base = [res_base_reg[n]['R²'] for n in nombres_reg]
r2_opt  = [res_opt_reg[n]['R²']  for n in nombres_reg]
x2 = range(len(nombres_reg))
axes[1].bar([i - 0.2 for i in x2], r2_base, width=0.4, label='Base',      color='mediumseagreen')
axes[1].bar([i + 0.2 for i in x2], r2_opt,  width=0.4, label='Optimizado', color='mediumpurple')
axes[1].set_xticks(list(x2))
axes[1].set_xticklabels(nombres_reg, rotation=15)
axes[1].set_title('R²: Base vs. Optimizado', fontweight='bold')
axes[1].set_ylabel('R²')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Serialización de Modelos Finales

In [ ]:
os.makedirs('models', exist_ok=True)

for nombre, modelo in mejores_clf.items():
    ruta = f"models/best_{nombre.lower()}_classifier.joblib"
    joblib.dump(modelo, ruta)
    print(f"  Guardado: {ruta}")

for nombre, modelo in mejores_reg.items():
    ruta = f"models/best_{nombre.lower()}_regressor.joblib"
    joblib.dump(modelo, ruta)
    print(f"  Guardado: {ruta}")

print("\nTodos los modelos serializados correctamente.")